# Phase 5 — Verifier & evaluation

Runs the golden set through the full graph, then the injection demo — the two headline artefacts for the project: abstention accuracy / citation coverage, and the before/after hallucinated-figure count.

In [ ]:
import subprocess, sys, os
REPO_URL = "https://github.com/<your-username>/stance.git"  # TODO: set this
if not os.path.isdir("/kaggle/working/stance"):
    subprocess.run(["git", "clone", REPO_URL, "/kaggle/working/stance"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "/kaggle/working/stance[retrieval,agents]"], check=True)
sys.path.insert(0, "/kaggle/working/stance")

In [ ]:
from stance.db import get_engine
import sys
sys.path.insert(0, "/kaggle/working/stance/eval")
from run_eval import run_golden_set, run_injection_demo, print_summary

engine = get_engine()

In [ ]:
golden_results = run_golden_set(engine)
for r in golden_results:
    print(f"[{r['kind']:>12}] {r['id']}: {r['question'][:60]:<60} -> {str(r.get('answer'))[:80]}")

## Injection demo — the headline security result

Simulates a writer that WAS successfully manipulated into citing a fabricated fact_id. The verifier re-derives every cited fact_id against `xbrl_facts`; a fabricated one resolves to nothing and is stripped.

In [ ]:
injection_result = run_injection_demo(engine)
print("BEFORE verification (what an unverified pipeline would publish):")
print(f"  {injection_result['draft']}\n")
print("AFTER verification:")
print(f"  {injection_result['cleaned_answer']}\n")
print(f"Verifier report: {injection_result['verifier_report']}")

## Headline summary table

In [ ]:
print_summary(golden_results, injection_result)